# SQL Analysis

## Project Goal

The goal of this notebook is to demonstrate how SQL can be used to analyze the marketing A/B testing dataset.

SQLite is used as a lightweight database engine inside Google Colab.

The analysis includes:

- basic dataset statistics;
- conversion analysis;
- group comparison;
- ad exposure analysis;
- day and hour performance;
- window functions.

In [1]:
import pandas as pd
import sqlite3

In [3]:
df = pd.read_csv(file_path)

df.head()

,unnamed:_0,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


## Create SQLite Database

The dataset is loaded into a SQLite database so that SQL queries can be executed directly in Google Colab.

In [4]:
conn = sqlite3.connect("marketing_ab_test.db")

df.to_sql("marketing_ab_test", conn, if_exists="replace", index=False)

print("Database successfully created.")

Database successfully created.


## Check Database Tables

In [5]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql(query, conn)

,name
0,marketing_ab_test


# SQL Queries

## Basic Statistics

This section provides a high-level overview of the dataset.

In [6]:
query = """
SELECT
    COUNT(*) AS total_users
FROM marketing_ab_test;
"""

pd.read_sql(query, conn)

,total_users
0,588101


In [7]:
query = """
SELECT
    COUNT(DISTINCT user_id) AS unique_users
FROM marketing_ab_test;
"""

pd.read_sql(query, conn)

,unique_users
0,588101


In [8]:
query = """
SELECT
    COUNT(*) AS converted_users
FROM marketing_ab_test
WHERE converted = 1;
"""

pd.read_sql(query, conn)

,converted_users
0,14843


In [9]:
query = """
SELECT
    ROUND(AVG(converted) * 100, 2) AS overall_conversion_rate
FROM marketing_ab_test;
"""

pd.read_sql(query, conn)

,overall_conversion_rate
0,2.52


## Group Analysis

This section compares the advertising group and the control group.

In [10]:
query = """
SELECT
    test_group,
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(AVG(converted) * 100, 2) AS conversion_rate
FROM marketing_ab_test
GROUP BY test_group
ORDER BY conversion_rate DESC;
"""

pd.read_sql(query, conn)

,test_group,users,conversions,conversion_rate
0,ad,564577,14423,2.55
1,psa,23524,420,1.79


In [11]:
query = """
SELECT
    test_group,
    ROUND(AVG(total_ads), 2) AS avg_total_ads,
    MIN(total_ads) AS min_ads,
    MAX(total_ads) AS max_ads
FROM marketing_ab_test
GROUP BY test_group;
"""

pd.read_sql(query, conn)

,test_group,avg_total_ads,min_ads,max_ads
0,ad,24.82,1,2065
1,psa,24.76,1,907


## Conversion by Day

This section analyzes conversion rates by the day when users saw the highest number of ads.

In [12]:
query = """
SELECT
    most_ads_day,
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(AVG(converted) * 100, 2) AS conversion_rate
FROM marketing_ab_test
GROUP BY most_ads_day
ORDER BY conversion_rate DESC;
"""

pd.read_sql(query, conn)

,most_ads_day,users,conversions,conversion_rate
0,Monday,87073,2857,3.28
1,Tuesday,77479,2312,2.98
2,Wednesday,80908,2018,2.49
3,Sunday,85391,2090,2.45
4,Friday,92608,2057,2.22
5,Thursday,82982,1790,2.16
6,Saturday,81660,1719,2.11


In [13]:
query = """
SELECT
    most_ads_day,
    test_group,
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(AVG(converted) * 100, 2) AS conversion_rate
FROM marketing_ab_test
GROUP BY most_ads_day, test_group
ORDER BY most_ads_day, test_group;
"""

pd.read_sql(query, conn)

,most_ads_day,test_group,users,conversions,conversion_rate
0,Friday,ad,88805,1995,2.25
1,Friday,psa,3803,62,1.63
2,Monday,ad,83571,2778,3.32
3,Monday,psa,3502,79,2.26
4,Saturday,ad,78802,1679,2.13
5,Saturday,psa,2858,40,1.40
6,Sunday,ad,82332,2027,2.46
7,Sunday,psa,3059,63,2.06
8,Thursday,ad,79077,1711,2.16
9,Thursday,psa,3905,79,2.02


In [14]:
query = """
SELECT
    most_ads_day,
    test_group,
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(AVG(converted) * 100, 2) AS conversion_rate
FROM marketing_ab_test
GROUP BY most_ads_day, test_group
ORDER BY most_ads_day, test_group;
"""

pd.read_sql(query, conn)

,most_ads_day,test_group,users,conversions,conversion_rate
0,Friday,ad,88805,1995,2.25
1,Friday,psa,3803,62,1.63
2,Monday,ad,83571,2778,3.32
3,Monday,psa,3502,79,2.26
4,Saturday,ad,78802,1679,2.13
5,Saturday,psa,2858,40,1.40
6,Sunday,ad,82332,2027,2.46
7,Sunday,psa,3059,63,2.06
8,Thursday,ad,79077,1711,2.16
9,Thursday,psa,3905,79,2.02


## Conversion by Hour

This section analyzes hourly conversion patterns.

In [15]:
query = """
SELECT
    most_ads_hour,
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(AVG(converted) * 100, 2) AS conversion_rate
FROM marketing_ab_test
GROUP BY most_ads_hour
ORDER BY most_ads_hour;
"""

pd.read_sql(query, conn)

,most_ads_hour,users,conversions,conversion_rate
0,0,5536,102,1.84
1,1,4802,62,1.29
2,2,5333,39,0.73
3,3,2679,28,1.05
4,4,722,11,1.52
5,5,765,16,2.09
6,6,2068,46,2.22
7,7,6405,116,1.81
8,8,17627,344,1.95
9,9,31004,595,1.92


In [16]:
query = """
SELECT
    most_ads_hour,
    test_group,
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(AVG(converted) * 100, 2) AS conversion_rate
FROM marketing_ab_test
GROUP BY most_ads_hour, test_group
ORDER BY most_ads_hour, test_group;
"""

pd.read_sql(query, conn)

,most_ads_hour,test_group,users,conversions,conversion_rate
0,0,ad,5309,102,1.92
1,0,psa,227,0,0.00
2,1,ad,4615,62,1.34
3,1,psa,187,0,0.00
4,2,ad,5152,39,0.76
5,2,psa,181,0,0.00
6,3,ad,2590,27,1.04
7,3,psa,89,1,1.12
8,4,ad,694,11,1.59
9,4,psa,28,0,0.00


## Ad Exposure Analysis

This section examines the relationship between the number of ads shown and conversion.

In [17]:
query = """
SELECT
    converted,
    COUNT(*) AS users,
    ROUND(AVG(total_ads), 2) AS avg_total_ads,
    MIN(total_ads) AS min_ads,
    MAX(total_ads) AS max_ads
FROM marketing_ab_test
GROUP BY converted;
"""

pd.read_sql(query, conn)

,converted,users,avg_total_ads,min_ads,max_ads
0,0,573258,23.29,1,2065
1,1,14843,83.89,1,1778


In [18]:
query = """
SELECT
    test_group,
    converted,
    COUNT(*) AS users,
    ROUND(AVG(total_ads), 2) AS avg_total_ads
FROM marketing_ab_test
GROUP BY test_group, converted
ORDER BY test_group, converted;
"""

pd.read_sql(query, conn)

,test_group,converted,users,avg_total_ads
0,ad,0,550154,23.27
1,ad,1,14423,83.91
2,psa,0,23104,23.70
3,psa,1,420,83.28


## Ad Frequency Segments

Users are grouped by the number of ads they saw. This helps identify whether conversion rate changes with ad exposure.

In [19]:
query = """
SELECT
    CASE
        WHEN total_ads BETWEEN 0 AND 10 THEN '0-10 ads'
        WHEN total_ads BETWEEN 11 AND 50 THEN '11-50 ads'
        WHEN total_ads BETWEEN 51 AND 100 THEN '51-100 ads'
        WHEN total_ads BETWEEN 101 AND 200 THEN '101-200 ads'
        ELSE '200+ ads'
    END AS ads_segment,
    COUNT(*) AS users,
    SUM(converted) AS conversions,
    ROUND(AVG(converted) * 100, 2) AS conversion_rate
FROM marketing_ab_test
GROUP BY ads_segment
ORDER BY conversion_rate DESC;
"""

pd.read_sql(query, conn)

,ads_segment,users,conversions,conversion_rate
0,101-200 ads,17112,2972,17.37
1,200+ ads,5952,927,15.57
2,51-100 ads,46002,5242,11.40
3,11-50 ads,258260,4844,1.88
4,0-10 ads,260775,858,0.33


## Window Function: Ranking Days by Conversion Rate

This query ranks days of the week by conversion rate using a window function.

In [20]:
query = """
SELECT
    most_ads_day,
    users,
    conversions,
    conversion_rate,
    RANK() OVER(ORDER BY conversion_rate DESC) AS conversion_rank
FROM (
    SELECT
        most_ads_day,
        COUNT(*) AS users,
        SUM(converted) AS conversions,
        ROUND(AVG(converted) * 100, 2) AS conversion_rate
    FROM marketing_ab_test
    GROUP BY most_ads_day
);
"""

pd.read_sql(query, conn)

,most_ads_day,users,conversions,conversion_rate,conversion_rank
0,Monday,87073,2857,3.28,1
1,Tuesday,77479,2312,2.98,2
2,Wednesday,80908,2018,2.49,3
3,Sunday,85391,2090,2.45,4
4,Friday,92608,2057,2.22,5
5,Thursday,82982,1790,2.16,6
6,Saturday,81660,1719,2.11,7


## Window Function: Ranking Hours by Conversion Rate

This query ranks hours by conversion rate.

In [21]:
query = """
SELECT
    most_ads_hour,
    users,
    conversions,
    conversion_rate,
    RANK() OVER(ORDER BY conversion_rate DESC) AS conversion_rank
FROM (
    SELECT
        most_ads_hour,
        COUNT(*) AS users,
        SUM(converted) AS conversions,
        ROUND(AVG(converted) * 100, 2) AS conversion_rate
    FROM marketing_ab_test
    GROUP BY most_ads_hour
)
ORDER BY conversion_rank
LIMIT 10;
"""

pd.read_sql(query, conn)

,most_ads_hour,users,conversions,conversion_rate,conversion_rank
0,16,37567,1156,3.08,1
1,20,28923,862,2.98,2
2,15,44683,1325,2.97,3
3,21,29976,867,2.89,4
4,17,34988,987,2.82,5
5,14,45648,1281,2.81,6
6,18,32323,885,2.74,7
7,19,30352,811,2.67,8
8,22,26432,690,2.61,9
9,13,47655,1176,2.47,10


## Group Difference by Day

This query compares advertising and control conversion rates by day.

In [22]:
query = """
WITH group_day AS (
    SELECT
        most_ads_day,
        test_group,
        ROUND(AVG(converted) * 100, 2) AS conversion_rate
    FROM marketing_ab_test
    GROUP BY most_ads_day, test_group
)

SELECT
    ad.most_ads_day,
    ad.conversion_rate AS ad_conversion_rate,
    psa.conversion_rate AS psa_conversion_rate,
    ROUND(ad.conversion_rate - psa.conversion_rate, 2) AS difference
FROM group_day ad
JOIN group_day psa
    ON ad.most_ads_day = psa.most_ads_day
WHERE ad.test_group = 'ad'
  AND psa.test_group = 'psa'
ORDER BY difference DESC;
"""

pd.read_sql(query, conn)

,most_ads_day,ad_conversion_rate,psa_conversion_rate,difference
0,Tuesday,3.04,1.44,1.60
1,Monday,3.32,2.26,1.06
2,Wednesday,2.54,1.58,0.96
3,Saturday,2.13,1.40,0.73
4,Friday,2.25,1.63,0.62
5,Sunday,2.46,2.06,0.40
6,Thursday,2.16,2.02,0.14


## Group Difference by Hour

This query compares advertising and control conversion rates by hour.

In [23]:
query = """
WITH group_hour AS (
    SELECT
        most_ads_hour,
        test_group,
        ROUND(AVG(converted) * 100, 2) AS conversion_rate
    FROM marketing_ab_test
    GROUP BY most_ads_hour, test_group
)

SELECT
    ad.most_ads_hour,
    ad.conversion_rate AS ad_conversion_rate,
    psa.conversion_rate AS psa_conversion_rate,
    ROUND(ad.conversion_rate - psa.conversion_rate, 2) AS difference
FROM group_hour ad
JOIN group_hour psa
    ON ad.most_ads_hour = psa.most_ads_hour
WHERE ad.test_group = 'ad'
  AND psa.test_group = 'psa'
ORDER BY difference DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,most_ads_hour,ad_conversion_rate,psa_conversion_rate,difference
0,6,2.32,0.00,2.32
1,5,2.16,0.00,2.16
2,0,1.92,0.00,1.92
3,4,1.59,0.00,1.59
4,1,1.34,0.00,1.34
5,20,3.03,1.76,1.27
6,14,2.86,1.61,1.25
7,7,1.85,0.84,1.01
8,22,2.65,1.64,1.01
9,23,2.30,1.29,1.01


# SQL Summary

Using SQL, we analyzed the marketing A/B testing dataset from several perspectives:

- overall dataset size;
- conversion rate;
- conversion by experimental group;
- ad exposure patterns;
- conversion by day and hour;
- ad frequency segments;
- group differences;
- window functions for ranking days and hours.

The SQL analysis supports the statistical A/B test by providing additional context about user behavior and campaign performance.